# Nyra IT V6 — finale del run 004, recupero ed export

Versione consolidata delle celle finali eseguite il 6 novembre 2026. Riprende i pesi completati da Drive, senza generare audio né riavviare training. Richiede il backup esistente del run `nyra-it-v6-004-continuous`, il dataset e `shutdown_backup_verified.json`.

Due modelli italiani: Nira e Ehi/Ei/Hei/Hey Nira, anche unite. Campioni sintetici già approvati automaticamente. Python 3.12 isolato, trainer e frontend ufficiali fissati. WAV su `/content/nyra_v6`, backup su Drive.

Il profilo finale privilegia il riconoscimento dei positivi: il gate originario di 0,5 falsi positivi/ora NON è superato. Le soglie sono scelte solo sulla validation; test separato descrittivo. Nessuna garanzia sulla voce umana. L'export non installa firmware. Non usare Esegui tutte sul vecchio notebook per riprendere questo run.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, shutil, hashlib, importlib.util
RUN_ID = 'nyra-it-v6-004-continuous'
SMOKE = False
USE_DRIVE = True
BACKUP_FOLDER = 'wakeword_training_nyra_it_v4'
RESUME = False  # True solo per un training interrotto dello stesso run.
ROOT = Path('/content/nyra_v6') / RUN_ID
ROOT.mkdir(parents=True, exist_ok=True)
VENV = Path('/content/nyra_v4_env')
PYTHON = str(VENV / 'bin/python')
assert RUN_ID and all(c.isalnum() or c in '-_' for c in RUN_ID)
print('Python Colab:', sys.version)
print('Disco libero:', round(shutil.disk_usage('/content').free/2**30,1), 'GiB')


## Ambiente fissato Python 3.12

In [ ]:
# Python del kernel e Python del training sono indipendenti.
# Usa sempre 3.12 per conservare le dipendenze TensorFlow previste.
VENV = Path('/content/nyra_v4_env_py312')
PYTHON = str(VENV / 'bin/python')
if not Path(PYTHON).exists():
    if sys.version_info[:2] == (3,12):
        subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
    else:
        UV_DIR = Path('/content/nyra_uv_bootstrap')
        UV = str(UV_DIR / 'bin/uv')
        if not Path(UV).exists():
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(UV_DIR),
                            'uv==0.9.7'], check=True)
        uv_env = os.environ.copy()
        uv_env['UV_PYTHON_INSTALL_DIR'] = '/content/nyra_python_managed'
        uv_env['UV_CACHE_DIR'] = '/content/nyra_uv_cache'
        subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)],
                       check=True, env=uv_env)
training_version = subprocess.check_output(
    [PYTHON, '-c', 'import sys; print(str(sys.version_info.major)+"."+str(sys.version_info.minor))'],
    text=True).strip()
if training_version != '3.12':
    raise RuntimeError('Ambiente training inatteso: ' + training_version + '. Usa un nuovo runtime.')
print('Python kernel:', sys.version.split()[0], '| Python training:', training_version)
requirements = [
    'tensorflow[and-cuda]==2.19.1', 'keras==3.10.0', 'numpy==2.1.3', 'scipy==1.15.3',
    'piper-tts==1.8.0', 'pymicro-features==2.0.2', 'mmap-ninja==0.9.0',
    'audiomentations==0.43.1', 'datasets==3.6.0', 'ai-edge-litert==1.4.0',
    'huggingface-hub==0.34.4', 'pyyaml==6.0.2', 'webrtcvad-wheels==2.0.14',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
    'git+https://github.com/OHF-Voice/micro-wake-word@4665173cd35f1cff9a61e06fc427f124766c488e',
]
req_hash = hashlib.sha256(json.dumps(requirements).encode()).hexdigest()
stamp = VENV / 'nyra_requirements.sha256'
if not stamp.exists() or stamp.read_text() != req_hash:
    subprocess.run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
    subprocess.run([PYTHON, '-m', 'pip', 'install'] + requirements, check=True)
    subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
    stamp.write_text(req_hash)
freeze = subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True)
(ROOT / 'environment.txt').write_text(freeze)
subprocess.run(['nvidia-smi'], check=False)
print('Ambiente isolato pronto.')

# Expose pip-installed CUDA libraries to every isolated training subprocess.
site_dir=Path(subprocess.check_output([PYTHON,'-c','import site; print(site.getsitepackages()[0])'],text=True).strip())
CUDA_LIBS=[str(p) for p in (site_dir/'nvidia').glob('*/lib')]
os.environ['LD_LIBRARY_PATH']=os.pathsep.join(CUDA_LIBS+[os.environ.get('LD_LIBRARY_PATH','')])
os.environ['PATH']=str(site_dir/'nvidia/cuda_nvcc/bin')+os.pathsep+os.environ['PATH']

# Kokoro CPU: avoids a second CUDA stack and GPU competition with TensorFlow.
subprocess.run([PYTHON,'-m','pip','install','torch==2.8.0','--index-url','https://download.pytorch.org/whl/cpu'],check=True)
subprocess.run([PYTHON,'-m','pip','install','kokoro==0.9.4'],check=True)
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','espeak-ng'],check=True)
subprocess.run([PYTHON,'-m','pip','check'],check=True)
(ROOT/'environment.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))


## Collegamento Drive

In [ ]:
from google.colab import drive
drive.mount('/content/nyra_drive')


## Recupero dei pesi finali e valutazione
Non esegue generate, features o train. Il worker approvato viene letto dal backup e non riscritto.

In [ ]:
# Recupero da checkpoint: WAV e feature sul disco Colab, pesi conservati.
from pathlib import Path
import os, shutil, subprocess, importlib.util, time, tarfile, json
RUN_ID='nyra-it-v6-004-continuous'
original=Path('/content/nyra_drive/MyDrive/wakeword_training_nyra_it_v4')/RUN_ID
assert (original/'approval.json').is_file()
ROOT=Path('/content/nyra_v6')/RUN_ID
assert not str(ROOT).startswith('/content/nyra_drive/')
ROOT.mkdir(parents=True,exist_ok=True)
WORKER=ROOT/'nyra_v6_worker.py'
for name in ['nyra_v6_worker.py','config.json','approval.json','preview.json','unavailable_preview_families.json']:
    dst=ROOT/name
    if not dst.exists():shutil.copy2(original/name,dst)
assert 'duration_attempt' in WORKER.read_text(), 'Il backup deve includere la correzione durata'
if not (ROOT/'preview').exists():shutil.copytree(original/'preview',ROOT/'preview')
latest=original.parent/'recovery_snapshots'/RUN_ID
assert (latest/'shutdown_backup_verified.json').is_file(), 'Salvataggio finale mancante'
for phrase in ['ehi_nira_it','nira_it']:
    assert (latest/'models'/phrase/'complete.json').is_file(), 'Training non completo: '+phrase
assert (latest/'nyra_v6_worker.py').read_bytes()==WORKER.read_bytes(), 'Worker diverso: fermarsi prima del recupero'
if not (ROOT/'models').exists():shutil.copytree(latest/'models',ROOT/'models')
print('RECUPERO PESI FINALI; GENERAZIONE E TRAINING ESCLUSI:',latest,flush=True)
if (latest/'scores').exists() and not (ROOT/'scores').exists():shutil.copytree(latest/'scores',ROOT/'scores')
print('Pesi esistenti conservati; directory di lavoro locale:', ROOT, flush=True)
spec=importlib.util.spec_from_file_location('nyra_v6_worker',WORKER)
worker=importlib.util.module_from_spec(spec);spec.loader.exec_module(worker)
worker.Run(ROOT).approved()
PYTHON='/content/nyra_v4_env_py312/bin/python'
ENV=os.environ.copy()
MWW_SOURCE=Path('/content/nyra_microwakeword_source')
if not (MWW_SOURCE/'.git').exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/OHF-Voice/micro-wake-word',str(MWW_SOURCE)],check=True)
subprocess.run(['git','-C',str(MWW_SOURCE),'checkout','--detach',worker.MWW_COMMIT],check=True)
assert (MWW_SOURCE/'microwakeword/audio/audio_utils.py').exists(), 'Sorgente ufficiale mancante'
ENV['PYTHONPATH']=str(MWW_SOURCE)+os.pathsep+ENV.get('PYTHONPATH','')
ENV['NYRA_BACKUP']=str(original.parent/'recovery_snapshots')
def stage(action,*args):
    with subprocess.Popen([PYTHON,'-u',str(WORKER),action,'--root',str(ROOT),*args],env=ENV,
                          stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:print(line,end='',flush=True)
        status=proc.wait()
    if status:raise RuntimeError(f'{action}: codice {status}; consultare log sopra')
subprocess.run([PYTHON,'-c',"import sys;sys.path.insert(0,sys.argv[1]);import nyra_v6_worker as w;w.Run(sys.argv[1]).ensure_voices()",str(ROOT)],check=True,env=ENV)
subprocess.run([PYTHON,'-c',"import tensorflow as tf,sys; c=tf.train.latest_checkpoint(sys.argv[1]); assert c; r=tf.train.load_checkpoint(c); print('CHECKPOINT LEGGIBILE',c,len(r.get_variable_to_shape_map()))",str(ROOT/'models/ehi_nira_it/restore')],check=True,env=ENV)
print('APPROVAZIONI VALIDE; CHECKPOINT LEGGIBILE; SSD COLAB:',ROOT,flush=True)
archive=original/'recovery_dataset_v1.tar.gz'
assert archive.is_file(), 'Archivio dataset assente: non rigeneriamo alla cieca'
if not (ROOT/'dataset.json').exists() and archive.exists():
    print('Ripristino audio sintetici dal backup completo...',flush=True)
    with tarfile.open(archive,'r:gz') as tf:
        for member in tf.getmembers():
            parts=Path(member.name).parts
            if not parts or parts[0] not in ('audio','dataset.json') or member.issym() or member.islnk():
                raise RuntimeError('Archivio dataset non valido')
        tf.extractall(ROOT,filter='data')
if (ROOT/'dataset.json').exists():
    rows=json.loads((ROOT/'dataset.json').read_text())['rows']
    missing=[r['path'] for r in rows if not (ROOT/r['path']).is_file()]
    if missing:raise RuntimeError(f'Dataset incompleto: {len(missing)} WAV mancanti. Nessun modello cancellato.')
def backup_dataset():
    if archive.exists():return
    manifest=ROOT/'dataset.json'
    if not manifest.exists():return
    rows=json.loads(manifest.read_text())['rows']
    size=sum((ROOT/r['path']).stat().st_size for r in rows)
    if shutil.disk_usage(original).free < size+2*2**30:
        raise RuntimeError('Spazio Drive insufficiente per proteggere il dataset: dati locali e pesi conservati.')
    print('Salvataggio archivio audio su Drive prima del training...',flush=True)
    tmp=archive.with_suffix('.gz.partial')
    with tarfile.open(tmp,'w:gz',compresslevel=1) as tf:
        tf.add(manifest,arcname='dataset.json')
        tf.add(ROOT/'audio',arcname='audio')
    tmp.replace(archive)
    print('Dataset protetto su Drive:',round(archive.stat().st_size/2**30,2),'GiB',flush=True)
# Training conclusi: valutazione estrae le proprie feature direttamente dai WAV.
steps=[('score',['--split','validation']),('calibrate',[]),('score',['--split','testing']),('report',[])]
try:
    for action,args in steps:
        worker.save(ROOT/'progress.json',dict(status='running',stage=action,args=args,updated=time.time()))
        print('FASE RECUPERO:',action,*args,flush=True)
        stage(action,*args)
        if action=='generate':backup_dataset()
        if (ROOT/'scores').exists():shutil.copytree(ROOT/'scores',latest/'scores',dirs_exist_ok=True)
        stage('backup')
    worker.save(ROOT/'progress.json',dict(status='complete',updated=time.time()))
except Exception as exc:
    worker.save(ROOT/'progress.json',dict(status='stopped',error=str(exc),updated=time.time()));raise
finally:stage('backup')


## Profilo con priorità ai positivi
Calibrazione su validation, valutazione separata su test. La normalizzazione della riga end=100 ripara soltanto un ritorno a capo introdotto nella copia dall’editor.

In [ ]:
# Profilo candidato scelto SOLO su validation; originali conservati.
import json, collections, shutil
ops=json.loads((ROOT/'operating_points.json').read_text())
candidates={}
for phrase,op in ops.items():
    good=[r for r in op['grid'] if r['recall']>=.99 and min(r['recall_by_condition'].values())>=.95 and r['window']==3]
    assert good, phrase
    chosen=min(good,key=lambda r:(r['faph'],r['hard_fpr'],-r['recall']))
    candidates[phrase]={k:chosen[k] for k in ['cutoff','window','recall','faph','hard_fpr']}
print('CANDIDATI FISSATI SU VALIDATION',json.dumps(candidates),flush=True)
results={}
for phrase,op in candidates.items():
    rows=json.loads((ROOT/'scores'/f'{phrase}_testing.json').read_text())['rows']
    positives=[r for r in rows if r['kind']=='positive' and r['phrase']==phrase]
    events=lambda r: worker.event_times(r['probs'],op['cutoff'],op['window'])
    hit=lambda r:any(r.get('start',4.)<=t<=r.get('end',100.)+1. for t in events(r))
    ambient=[r for r in rows if r['kind']=='ambient']
    hard=[r for r in rows if r['kind']=='hard']
    hours=sum(r['duration'] for r in ambient)/3600
    conditions=sorted({r['condition'] for r in positives})
    results[phrase]=dict(validation=op,test_recall=sum(map(hit,positives))/len(positives),test_faph=sum(len(events(r)) for r in ambient)/hours,hard_fpr=sum(bool(events(r)) for r in hard)/len(hard),min_condition=min(sum(hit(r) for r in positives if r['condition']==c)/sum(r['condition']==c for r in positives) for c in conditions))
    print(phrase,json.dumps(results[phrase]),flush=True)
worker.save(ROOT/'recall_priority_diagnostic.json',results)
shutil.copy2(ROOT/'recall_priority_diagnostic.json',latest/'recall_priority_diagnostic.json')
print('DIAGNOSI SALVATA; PESI E SOGLIE ORIGINALI INVARIATI',flush=True)


## Export finale separato e backup

In [ ]:
import json, shutil
from pathlib import Path
diag=json.loads((ROOT/'recall_priority_diagnostic.json').read_text())
run=worker.Run(ROOT);run.approved()
out=ROOT/'export_recall_priority';out.mkdir(exist_ok=True)
report=json.loads((ROOT/'evaluation.json').read_text())
for phrase,item in diag.items():
    assert item['test_recall']>=.99 and item['min_condition']>=.95
    src=run.model_path(phrase);assert worker.file_hash(src)==report['model_hashes'][phrase]
    dst=out/f'{phrase}.tflite';shutil.copy2(src,dst);assert worker.file_hash(dst)==worker.file_hash(src)
    op=item['validation']
    worker.save(out/f'{phrase}.json',dict(type='micro',wake_word=worker.PHRASES[phrase],author=run.cfg['author'],model=dst.name,trained_languages=['it'],version=2,micro=dict(probability_cutoff=op['cutoff'],sliding_window_size=op['window'],feature_step_size=10,tensor_arena_size=run.cfg['tensor_arena_estimate'],minimum_esphome_version='2024.7.0')))
    print('EXPORT VERIFICATO',phrase,worker.file_hash(dst),'soglia',op['cutoff'],'finestra',op['window'],flush=True)
worker.save(out/'recall_priority_diagnostic.json',diag)

worker.save(out/'provenance.json',dict(run_id=RUN_ID,policy='recall_priority_user_requested',original_false_positive_gate_passed=False,threshold_selection='validation_only',human_tests_performed=False,arena_measured=False,model_hashes=report['model_hashes']))
shutil.copy2(ROOT/'evaluation.json',out/'evaluation_original_strict_profile.json')
(out/'README.txt').write_text('Profilo priorita positivi richiesto dall utente. Gate originale 0.5 FP/h NON superato. Solo test sintetici. Arena stimata. Worker e pesi invariati.')
archive=Path(shutil.make_archive(str(ROOT/'nyra_it_v6_recall_priority'),'zip',out))
shutil.copytree(out,latest/'export_recall_priority',dirs_exist_ok=True)
shutil.copy2(archive,latest/archive.name)
print('EXPORT COMPLETATO E SALVATO SU DRIVE',archive.name,archive.stat().st_size,flush=True)


In [ ]:
from google.colab import files
files.download(str(ROOT/'nyra_it_v6_recall_priority.zip'))


## Verifiche effettuate
I modelli esportati sono stati compilati con ESPHome 2026.10.1 e installati su Soggiorno e Mansarda il 6 novembre 2026; verificati avvio e stato running per 65 secondi ciascuno. Non eseguito qui un nuovo training completo né test umani di riconoscimento. Vedere v6-004/DEPLOYMENT_2026-11-06.json e i report sintetici.